---
---
# 🏗️ Self-Assignment — Mini RAG Project

> **Complete this after class.** Build a small but complete RAG system on a document of your choice, following the same architecture demonstrated in the session.

---

## 🎯 Project Overview

You will build a **Document Q&A Assistant** that:
1. Loads and chunks **your own document** (PDF, TXT, or plain text)
2. Indexes it into ChromaDB using Azure embeddings
3. Answers questions using a **grounded RAG pipeline**
4. Supports **hybrid search** (BM25 + dense)
5. Returns **cited, structured answers**
6. Handles failure modes gracefully

---

## 📐 Architecture You Are Building

```
Your Document (PDF/TXT)
        │
   [Section 1] Load & chunk
        │
   [Section 2] Embed → ChromaDB
        │              │
   User Query ──► [Section 3] Hybrid Search (BM25 + Dense + RRF)
                       │
                  [Section 4] Build grounded prompt
                       │
                  Azure OpenAI (gpt-4o-mini)
                       │
                  [Section 5] Structured answer + citations
                       │
                  [Section 6] Evaluation & reflection
```

---

## 📋 Grading Rubric

| Section | Task | Points |
|---------|------|--------|
| 1 | Document loaded, chunked correctly, chunk stats printed | 15 |
| 2 | Vectors stored in ChromaDB, metadata tagged | 15 |
| 3 | Hybrid search working, BM25 + Dense + RRF fused | 20 |
| 4 | Grounded system prompt, 5 questions answered | 20 |
| 5 | Citation JSON returned and parsed correctly | 15 |
| 6 | At least 2 failure modes demonstrated with fixes | 15 |
| **Total** | | **100** |

---

## 🔖 How to Submit
1. Complete all `# TODO` cells below
2. Run the entire notebook from top to bottom (Runtime → Run All)
3. Save as: `RAG_Assignment_<YourName>.ipynb`
4. Upload to the class portal

> ⚠️ **Important:** All outputs must be visible when you submit. Do NOT clear outputs before submitting.

---
## Step 0 — Choose Your Document

Pick **one** of the following options for your source document.
Choose something you find interesting — you will be asking 5 questions about it!

| Option | Type | Example |
|--------|------|---------|
| A | Company annual/quarterly report | Apple Q4 2024, any public company 10-K |
| B | Research paper or article | Any PDF from arXiv, WHO, World Bank |
| C | Product documentation | API docs, user manual, technical spec |
| D | News article collection | 3–5 articles on the same topic as one text |
| E | Wikipedia article (long-form) | Copy a long Wikipedia article as plain text |

🔑 **Minimum length:** at least 800 words (~5,000 characters) so you get enough chunks to make retrieval meaningful.

In [1]:
from google import genai

In [2]:
from google.colab import userdata

# ── Google Gemini API credentials ──────────────────────────────────────────
# Find these in: Google AI Studio
GOOGLE_API_KEY    = userdata.get("GOOGLE_API_KEY")

# ── Model names ────────────────────────────────────────────────────────────
CHAT_MODEL      = "gemini-3.1-flash-lite" # or "gemini-1.5-pro"
EMBEDDING_MODEL = "text-embedding-001"

print(f"\n✅ Configuration saved")
print(f"   Chat Model      : {CHAT_MODEL}")
print(f"   Embedding Model : {EMBEDDING_MODEL}")


✅ Configuration saved
   Chat Model      : gemini-3.1-flash-lite
   Embedding Model : text-embedding-001


In [3]:
# ── Step 0: Paste or load your document ───────────────────────────────────
#
# Option A — paste text directly:
# MY_DOCUMENT = """
# <paste your document content here>
# """
#
# Option B — load from uploaded file (Google Colab):
# from google.colab import files
# uploaded = files.upload()             # will prompt you to pick a file
# filename = list(uploaded.keys())[0]
# MY_DOCUMENT = uploaded[filename].decode('utf-8')
#
# Option C — load a PDF (requires pypdf):
!pip install -q pypdf
from pypdf import PdfReader
reader = PdfReader("1506.02640v5.pdf")
MY_DOCUMENT = "\n".join(page.extract_text() for page in reader.pages)

# ── TODO: set MY_DOCUMENT to your chosen text ─────────────────────────────
# MY_DOCUMENT = """  # <-- REPLACE THIS with your document
# """

# ── TODO: describe your document ──────────────────────────────────────────
DOC_NAME    = "1506.02640v5.pdf"       # a short filename, e.g. 'apple_q4_2024.pdf'
DOC_TOPIC   = "You Only Look Once"  # e.g. 'Apple Q4 2024 earnings report'

# Validation
assert len(MY_DOCUMENT.strip()) >= 500, \
    f"Document too short ({len(MY_DOCUMENT)} chars). Minimum 800 words required."

print(f"✅ Document loaded: '{DOC_TOPIC}'")
print(f"   Name     : {DOC_NAME}")
print(f"   Size     : {len(MY_DOCUMENT):,} characters, ~{len(MY_DOCUMENT.split())} words")

   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 338.8/338.8 kB 10.3 MB/s eta 0:00:00
✅ Document loaded: 'You Only Look Once'
   Name     : 1506.02640v5.pdf
   Size     : 42,216 characters, ~7023 words


---
## Step 1 — Load & Chunk Your Document

Use the `recursive_split()` function from the demo session.

**Your task:**
- Choose `chunk_size` and `chunk_overlap` appropriate for your document type
- Print chunk statistics (count, average size, min, max)
- Display the first 3 chunks to verify the split looks sensible

**Guidance:**
- Dense prose (reports, articles): `chunk_size=400–512`
- Technical docs with lists: `chunk_size=250–350`
- Always use `chunk_overlap` ≈ 10–15% of `chunk_size`

In [4]:
def recursive_split(
    text: str,
    chunk_size: int = 400,
    chunk_overlap: int = 60,
    separators: list[str] = None,
) -> list[str]:
    """
    Splits text into chunks of at most `chunk_size` characters.
    Tries each separator in order; falls back to the next if chunks are still too large.
    Preserves `chunk_overlap` characters of context between consecutive chunks.
    """
    if separators is None:
        separators = ["\n\n", "\n", ". ", " ", ""]

    def _split(text: str, seps: list[str]) -> list[str]:
        # Base case: text is small enough
        if len(text) <= chunk_size:
            return [text.strip()] if text.strip() else []

        sep = seps[0] if seps else ""
        parts = text.split(sep) if sep else list(text)

        chunks, current = [], ""
        for part in parts:
            piece = (current + sep + part) if current else part
            if len(piece) <= chunk_size:
                current = piece
            else:
                if current:
                    chunks.append(current.strip())
                # If the part itself is too large, recurse with next separator
                if len(part) > chunk_size and len(seps) > 1:
                    chunks.extend(_split(part, seps[1:]))
                    current = ""
                else:
                    current = part
        if current:
            chunks.append(current.strip())
        return [c for c in chunks if c]

    raw_chunks = _split(text, separators)

    # Apply overlap: carry the last `chunk_overlap` chars of chunk[i] into chunk[i+1]
    if chunk_overlap == 0 or len(raw_chunks) <= 1:
        return raw_chunks

    overlapped = [raw_chunks[0]]
    for i in range(1, len(raw_chunks)):
        tail = overlapped[-1][-chunk_overlap:]
        overlapped.append((tail + " " + raw_chunks[i]).strip())
    return overlapped

In [5]:
# ── Step 1: Chunk your document ───────────────────────────────────────────
# The recursive_split() function is already defined above in the demo section.
# Just call it with appropriate parameters for your document type.

# ── TODO: Set chunk parameters ────────────────────────────────────────────
CHUNK_SIZE    = 500   # TODO: choose a value between 200 and 600
CHUNK_OVERLAP = 50   # TODO: choose overlap ≈ 10–15% of chunk_size

assert CHUNK_SIZE > 0,    "Set CHUNK_SIZE (e.g. 400)"
assert CHUNK_OVERLAP > 0, "Set CHUNK_OVERLAP (e.g. 60)"

# ── TODO: Split the document ──────────────────────────────────────────────
my_chunks = recursive_split(MY_DOCUMENT, chunk_size=CHUNK_SIZE, chunk_overlap=CHUNK_OVERLAP)  # TODO: call recursive_split() here

assert my_chunks is not None and len(my_chunks) >= 3, \
    "my_chunks must contain at least 3 chunks. Check your recursive_split() call."

# ── TODO: Print chunk statistics ─────────────────────────────────────────
# Expected output:
# ✅ Created X chunks
#    Avg size : ??? chars
#    Min size : ??? chars
#    Max size : ??? chars
print("TODO: print chunk statistics")
print(f"Avg size : {sum(len(c) for c in my_chunks) // len(my_chunks)} chars")
print(f"Min size : {min(len(c) for c in my_chunks)} chars")
print(f"Max size : {max(len(c) for c in my_chunks)} chars")
print()

# ── TODO: Display first 3 chunks ─────────────────────────────────────────
print("\nFirst 3 chunks:")
for i, c in enumerate(my_chunks[:3]):
    print(f"  {i+1}. {c[:100]}...")
# TODO: loop over my_chunks[:3] and print each one

TODO: print chunk statistics
Avg size : 523 chars
Min size : 412 chars
Max size : 551 chars


First 3 chunks:
  1. You Only Look Once:
Uniﬁed, Real-Time Object Detection
Joseph Redmon∗, Santosh Divvala∗†, Ross Girsh...
  2. problem to spatially separated bounding boxes and associated class probabilities. A single neural ne...
  3. rocesses an astounding 155 frames per second while still achieving double the mAP of other real-time...


---
## Step 2 — Embed & Index into ChromaDB

**Your task:**
- Embed all chunks using the `embed()` helper (Azure API)
- Create a new ChromaDB collection for your document
- Store chunks with meaningful **metadata** tags (at minimum: `source` and `section` or `chunk_index`)
- Run a test search to verify retrieval works

**Tip:** You can reuse `label_section()` from the demo, or write your own labelling logic for your document.

In [20]:
import json, math, re, textwrap, time
from pathlib import Path

client = genai.Client(api_key=GOOGLE_API_KEY)

# ── Helper: call chat completion ───────────────────────────────────────────
def chat(messages: list[dict], temperature: float = 0, max_tokens: int = 1024) -> str:
    """
    Thin wrapper around the Google Gemini chat completions API.
    messages = [{"role": "user"|"model", "parts": ["..."]}]
    Returns the assistant reply as a plain string.
    """
    # Adjust messages for Gemini format
    gemini_messages = []
    system_prefix = ""
    for msg in messages:
        if msg["role"] == "system":
            system_prefix = msg["content"] + "\n\n"
        elif msg["role"] == "user":
            content = system_prefix + msg["content"]
            system_prefix = ""
            gemini_messages.append({"role": "user", "parts": [{"text": content}]})
        elif msg["role"] == "assistant":
            gemini_messages.append({"role": "model", "parts": [{"text": msg["content"]}]})

    response = client.models.generate_content(
        model=CHAT_MODEL,
        contents=gemini_messages,
        config={
            "temperature": temperature,
            "max_output_tokens": max_tokens,
        }
    )
    return response.text


# ── Helper: embed a list of strings ───────────────────────────────────────
def embed(texts: list[str]) -> list[list[float]]:
    """
    Returns a list of embedding vectors (one per input text).
    Batches automatically — Gemini allows up to 2048 inputs per call.
    """
    BATCH = 64
    vectors = []
    for i in range(0, len(texts), BATCH):
        batch = texts[i : i + BATCH]
        response = client.models.embed_content(
            model='gemini-embedding-001',
            contents=batch,
        )
        vectors.extend([item.values for item in response.embeddings])
    return vectors


# ── Helper: pretty-print long text ────────────────────────────────────────
def pp(text: str, width: int = 88):
    print(textwrap.fill(str(text), width=width))


# Quick smoke-test
reply = chat([{"role": "user", "content": "Reply with exactly: Google Gemini is ready."}]
)
print(f"Smoke test → {reply}")

Smoke test → Google Gemini is ready.


In [21]:
# ── Step 2a: Embed all chunks via Azure ───────────────────────────────────
import time

print(f"Embedding {len(my_chunks)} chunks...")

# ── TODO: Call embed() and store the result ───────────────────────────────
my_vectors = embed(my_chunks)  # TODO: call embed(my_chunks)

assert my_vectors is not None, "Call embed(my_chunks) and assign the result to my_vectors"
assert len(my_vectors) == len(my_chunks), "Number of vectors must match number of chunks"

print(f"✅ Embedded {len(my_vectors)} chunks")
print(f"   Vector dimensions : {len(my_vectors[0])}")
print(f"   Preview (chunk 0) : {[round(v, 4) for v in my_vectors[0][:5]]}...")

Embedding 89 chunks...
✅ Embedded 89 chunks
   Vector dimensions : 3072
   Preview (chunk 0) : [0.0058, -0.0192, 0.0171, -0.0566, -0.0218]...


In [23]:
# ── Step 2b: Store in ChromaDB ────────────────────────────────────────────
!pip install chromadb
import chromadb, shutil
from pathlib import Path

# Using /content/ instead of /tmp/ can be more reliable for persistence in Colab
MY_CHROMA_PATH = "/content/my_rag_chroma"
if Path(MY_CHROMA_PATH).exists():
    shutil.rmtree(MY_CHROMA_PATH)

# Re-initialize the client
my_chroma_client = chromadb.PersistentClient(path=MY_CHROMA_PATH)

class MyEmbeddingFn(chromadb.EmbeddingFunction):
    def __call__(self, input):
        return embed(input)

my_collection = my_chroma_client.create_collection(
    name="my_document",
    embedding_function=MyEmbeddingFn(),
    metadata={"hnsw:space": "cosine"},
)

def label_section(chunk: str) -> str:
    """Categorize research paper chunks based on keywords."""
    lower = chunk.lower()
    if "abstract" in lower: return "abstract"
    if "introduction" in lower or "related work" in lower: return "introduction"
    if "method" in lower or "architecture" in lower or "model" in lower: return "methodology"
    if "experiment" in lower or "results" in lower or "evaluation" in lower: return "experiments"
    if "conclusion" in lower or "future work" in lower: return "conclusion"
    if "reference" in lower: return "references"
    return "general"

# ── Helper: assign metadata to each chunk ───────
def my_metadata(chunk: str, index: int) -> dict:
    return {
        "source": DOC_NAME,
        "chunk_index": index,
        "section": label_section(chunk)
    }

# ── Add all chunks to the collection ────────────────────────────────
ids = [f"chunk_{i}" for i in range(len(my_chunks))]
metadatas = [my_metadata(chunk, i) for i, chunk in enumerate(my_chunks)]

my_collection.add(
    ids=ids,
    documents=my_chunks,
    embeddings=my_vectors,
    metadatas=metadatas,
)

assert my_collection.count() == len(my_chunks), \
    f"Expected {len(my_chunks)} items in collection, got {my_collection.count()}"

print(f"✅ Indexed {my_collection.count()} chunks into ChromaDB with section labels")

     ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 52.0/52.0 kB 2.4 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 23.3/23.3 MB 56.7 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 278.2/278.2 kB 23.7 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 2.0/2.0 MB 79.4 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 18.2/18.2 MB 63.2 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 71.8/71.8 kB 6.2 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 170.9/170.9 kB 12.9 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 61.3/61.3 kB 5.3 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 203.7/203.7 kB 16.6 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 71.6/71.6 kB 6.2 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 60.6/60.6 kB 4.5 MB/s eta 0:00:00
  Attempting uninstall: opentelemetry-proto
    Found existing installation: opentelemetry-proto 1.38.0
    Uninstalling opentelem

/tmp/ipykernel_1802/2493078822.py:20: DeprecationWarning: The class MyEmbeddingFn does not implement __init__. This will be required in a future version.
  embedding_function=MyEmbeddingFn(),


In [24]:
# ── Step 2c: Verify search works ─────────────────────────────────────────
# Build your own dense_search function that queries MY collection

def my_dense_search(query: str, k: int = 5, where: dict = None) -> list[dict]:
    """
    Search MY collection. Returns list of {text, score, metadata} dicts.
    """
    # ── TODO: implement this function ─────────────────────────────────────
    # Steps:
    #   1. embed([query]) → query_vector
    #   2. my_collection.query(query_embeddings=[query_vector], ...)
    #   3. return list of {text, score, metadata}
    # pass  # TODO: replace with your implementation
    query_vector = embed([query])[0]
    kwargs = dict(query_embeddings=[query_vector], n_results=k, include=["documents", "distances", "metadatas"])
    if where:
        kwargs["where"] = where
    results = my_collection.query(**kwargs)

    return [
        {
            "text": results["documents"][0][i],
            "score": 1 - results["distances"][0][i],   # cosine: distance→similarity
            "metadata": results["metadatas"][0][i],
        }
        for i in range(len(results["documents"][0]))
    ]


# ── TODO: Run a test search using a topic from your document ─────────────
TEST_QUERY = "What does YOLO stands for"  # TODO: fill in a relevant question about your document

assert TEST_QUERY, "Set TEST_QUERY to a question about your document"

test_results = my_dense_search(TEST_QUERY, k=3)

assert test_results and len(test_results) > 0, "my_dense_search() returned no results"

print(f"Test query: '{TEST_QUERY}'")
print()
for i, r in enumerate(test_results):
    print(f"  Rank #{i+1} | score={r['score']:.4f}")
    print(f"  {r['text'][:140]}...")
    print()

Test query: 'What does YOLO stands for'

  Rank #1 | score=0.6125
  edict what objects are
present and where they are. YOLO is refreshingly simple: see Figure 1. A sin-
gle convolutional network simultaneousl...

  Rank #2 | score=0.6113
  You Only Look Once:
Uniﬁed, Real-Time Object Detection
Joseph Redmon∗, Santosh Divvala∗†, Ross Girshick¶, Ali Farhadi∗†
University of Washin...

  Rank #3 | score=0.6056
  e introduce YOLO, a uniﬁed model for object detec- tion. Our model is simple to construct and can be trained
directly on full images. Unlike...



---
## Step 3 — Hybrid Search (BM25 + Dense + RRF)

**Your task:**
- Build a BM25 index over `my_chunks`
- Implement `my_bm25_search()` using `rank_bm25`
- Implement `my_hybrid_search()` that fuses BM25 + dense results using the `rrf_fusion()` function from the demo
- Run a comparison: show how BM25, Dense, and Hybrid differ on the same query

**Recall:** `rrf_fusion()` is already defined in the demo section above.

In [25]:
# ── Step 3a: Build BM25 index ─────────────────────────────────────────────
!pip install rank_bm25
from rank_bm25 import BM25Okapi
import re

def simple_tokenize(text: str) -> list[str]:
    return re.sub(r'[^\w\s]', ' ', text.lower()).split()

# ── TODO: Build the BM25 index over my_chunks ────────────────────────────
my_bm25_index = BM25Okapi([simple_tokenize(c) for c in my_chunks])  # TODO: BM25Okapi([simple_tokenize(c) for c in my_chunks])

assert my_bm25_index is not None, "Build the BM25 index"
print("✅ BM25 index built")

✅ BM25 index built


In [35]:
# ── Step 3b: Implement BM25 search ───────────────────────────────────────
def rrf_fusion(result_lists: list[list[dict]], k: int = 60, id_key: str = "text") -> list[dict]:
    """
    Merge multiple ranked result lists using Reciprocal Rank Fusion.
    Documents are identified by id_key (default: 'text' content).
    Higher rrf_score = better rank.
    """
    scores: dict[str, float] = {}
    docs:   dict[str, dict]  = {}

    for result_list in result_lists:
        for rank, doc in enumerate(result_list):
            doc_id = doc[id_key][:80]  # use first 80 chars as stable ID
            scores[doc_id] = scores.get(doc_id, 0.0) + 1.0 / (k + rank + 1)
            docs[doc_id] = doc

    merged = sorted(scores.keys(), key=lambda did: scores[did], reverse=True)
    results = []
    for doc_id in merged:
        entry = dict(docs[doc_id])
        entry["rrf_score"] = round(scores[doc_id], 6)
        results.append(entry)
    return results

def my_bm25_search(query: str, k: int = 5) -> list[dict]:
    """
    BM25 keyword search over my_chunks.
    Returns list of {text, score, metadata, chunk_index} dicts.
    """
    # ── TODO: implement this function ─────────────────────────────────────
    # Steps:
    #   1. tokenize the query with simple_tokenize()
    #   2. get scores: my_bm25_index.get_scores(tokens)
    #   3. sort indices by score descending, take top k
    #   4. return list of dicts with text, score, metadata, chunk_index
    # pass  # TODO: replace with your implementation
    tokens = simple_tokenize(query)
    scores = my_bm25_index.get_scores(tokens)
    top_indices = sorted(range(len(scores)), key=lambda i: scores[i], reverse=True)[:k]
    return [
        {
            "text": my_chunks[i],
            "score": float(scores[i]),
            "metadata": my_metadata(my_chunks[i], i), # Fixed metadata to use my_metadata function
            "chunk_index": i,
        }
        for i in top_indices
    ]


# ── Step 3c: Implement hybrid search ─────────────────────────────────────

def my_hybrid_search(query: str, k: int = 5, dense_k: int = 10, bm25_k: int = 10) -> list[dict]:
    """
    Hybrid search: BM25 + Dense + RRF fusion.
    Use rrf_fusion() from the demo section above.
    """
    # ── TODO: implement this function ─────────────────────────────────────
    # Steps:
    #   1. dense_hits  = my_dense_search(query, k=dense_k)
    #   2. bm25_hits   = my_bm25_search(query, k=bm25_k)
    #   3. fused       = rrf_fusion([dense_hits, bm25_hits])
    #   4. return fused[:k]
    # pass  # TODO: replace with your implementation
    dense_hits = my_dense_search(query, k=dense_k)
    bm25_hits  = my_bm25_search(query, k=bm25_k)
    fused      = rrf_fusion([dense_hits, bm25_hits])
    return fused[:k]


# ── TODO: Test with a query where exact keywords matter ──────────────────
KEYWORD_QUERY = "What is the reported processing speed of the YOLO model?"  # TODO: a query with specific terms/names from your document

assert KEYWORD_QUERY, "Set KEYWORD_QUERY"

print(f"Comparison for: '{KEYWORD_QUERY}'")
print()
d_top = my_dense_search(KEYWORD_QUERY, k=1)
b_top = my_bm25_search(KEYWORD_QUERY, k=1)
h_top = my_hybrid_search(KEYWORD_QUERY, k=1)

assert d_top and b_top and h_top, "All three search functions must return results"

print(f"  Dense  : {d_top[0]['text'][:100]}...")
print(f"  BM25   : {b_top[0]['text'][:100]}...")
print(f"  Hybrid : {h_top[0]['text'][:100]}...")

Comparison for: 'What is the reported processing speed of the YOLO model?'

  Dense  : ystem that runs in real-time (30 frames per second or better) [31]. We compare YOLO to their GPU imp...
  BM25   : tunately, this combination doesn’t beneﬁt from the speed of YOLO since we run each model seperately ...
  Hybrid : ystem that runs in real-time (30 frames per second or better) [31]. We compare YOLO to their GPU imp...


---
## Step 4 — Build the Grounded RAG Chain

**Your task:**
- Write a system prompt appropriate for your document topic
- Implement a `my_rag()` function that uses `my_hybrid_search()` for retrieval
- Answer **5 meaningful questions** about your document
- Test the refusal: ask one question whose answer is NOT in the document

**Good questions to ask:**
- A specific factual question (a number, date, or name from the document)
- A summary question ("What are the main points of section X?")
- A comparative question ("How does X compare to Y?")
- A causal question ("Why did X happen?")
- An implication question ("What does X mean for the future?")

In [27]:
# ── Step 4a: Write your system prompt ─────────────────────────────────────

# ── TODO: Write a domain-appropriate system prompt for your document ──────
# Adapt the one from the demo session, but tailor it to your document type.
# Required elements:
#   1. A role description ("You are a ... assistant")
#   2. Grounding rule: "Answer ONLY using the [CONTEXT] below"
#   3. Refusal rule: "If not in context, say 'I don't have enough information'"
#   4. Citation rule: "Cite source after each claim"

MY_SYSTEM_PROMPT = """
You are an expert Object Detection Research Assistant, specialized in the YOLO (You Only Look Once) framework.

Your task is to answer questions truthfully and concisely.

Answer ONLY using the [CONTEXT] below.
If the answer is not available in the provided context, state clearly 'I don't have enough information in the provided context to answer this question.'
Do NOT make up answers.

Cite the source after each claim in your response using the format (Source: <source_name> - Chunk <chunk_index>). For example: 'YOLO is a state-of-the-art object detection system (Source: 1506.02640v5.pdf - Chunk 5)'."""

assert len(MY_SYSTEM_PROMPT.strip()) > 50, \
    "MY_SYSTEM_PROMPT is too short. Write a real grounding prompt."

print("✅ System prompt set")
print(MY_SYSTEM_PROMPT)

✅ System prompt set

You are an expert Object Detection Research Assistant, specialized in the YOLO (You Only Look Once) framework. 

Your task is to answer questions truthfully and concisely.

Answer ONLY using the [CONTEXT] below. 
If the answer is not available in the provided context, state clearly 'I don't have enough information in the provided context to answer this question.' 
Do NOT make up answers.

Cite the source after each claim in your response using the format (Source: <source_name> - Chunk <chunk_index>). For example: 'YOLO is a state-of-the-art object detection system (Source: 1506.02640v5.pdf - Chunk 5)'.


In [28]:
# ── Step 4b: Implement my_rag() ───────────────────────────────────────────

def my_format_context(hits: list[dict]) -> str:
    """Format retrieved chunks into a readable context block."""
    # ── TODO: implement this helper ───────────────────────────────────────
    # For each hit, include: index number, source metadata, and chunk text
    # Separate chunks with a clear divider (e.g. '\n\n---\n\n')
    # pass  # TODO: replace
    parts = []
    for i, h in enumerate(hits):
        src = h['metadata'].get('source', 'unknown')
        sec = h['metadata'].get('section', 'unknown')
        parts.append(f"[{i+1}] Source: {src} | Section: {sec}\n{h['text']}")
    return "\n\n---\n\n".join(parts)


def my_rag(question: str, k: int = 5) -> dict:
    """
    Full RAG pipeline for your document.
    Returns: {question, answer, hits}
    """
    # ── TODO: implement this function ─────────────────────────────────────
    # Steps:
    #   1. hits    = my_hybrid_search(question, k=k)
    #   2. context = my_format_context(hits)
    #   3. messages = [{system prompt}, {user: context + question}]
    #   4. answer  = chat(messages)
    #   5. return {question, answer, hits}
    # pass  # TODO: replace
    hits = my_hybrid_search(question, k=k)
    context = my_format_context(hits)

    messages = [
        {"role": "system", "content": MY_SYSTEM_PROMPT},
        {"role": "user",   "content": f"[CONTEXT]\n{context}\n\n[QUESTION]\n{question}"},
    ]

    answer = chat(messages)

    return {"question": question, "answer": answer, "hits": hits}



print("✅ my_rag() defined")

✅ my_rag() defined


In [29]:
# ── Step 4c: Answer 5 questions about your document ───────────────────────

# ── TODO: Write 5 meaningful questions about your document ────────────────
MY_QUESTIONS = [
    "What is the reported processing speed of the YOLO model?",   # Question 1 — specific factual (number, date, name)
    "What is the main innovation introduced by the YOLO framework?",   # Question 2 — summary or main point
    "How does the YOLO detection system differ from traditional object detection systems like R-CNN regarding its approach to detection?",   # Question 3 — comparison or contrast
    "Why was the YOLO system developed as an alternative to existing object detection methods?",   # Question 4 — causal (why did X happen?)
    "What are the primary advantages of using YOLO for real-time object detection applications?"   # Question 5 — implication or outlook
]

assert all(q.strip() for q in MY_QUESTIONS), \
    "All 5 questions must be filled in (no empty strings)"

# Run all questions
for q in MY_QUESTIONS:
    result = my_rag(q)
    print(f"{'='*68}")
    print(f"Q: {q}")
    print(f"{'─'*68}")
    print(f"A: {result['answer']}")
    print(f"   [Retrieved {len(result['hits'])} chunks | top sim: {result['hits'][0].get('score', result['hits'][0].get('rrf_score', '?')):.4f}]")

Q: What is the reported processing speed of the YOLO model?
────────────────────────────────────────────────────────────────────
A: The YOLO system runs in real-time at 30 frames per second or better (Source: 1506.02640v5.pdf - Chunk 1). Additionally, the model processes 155 frames per second (Source: 1506.02640v5.pdf - Chunk 4).
   [Retrieved 5 chunks | top sim: 9.6837]
Q: What is the main innovation introduced by the YOLO framework?
────────────────────────────────────────────────────────────────────
A: The main innovation of the YOLO framework is that it reframes object detection as a single regression problem, mapping image pixels directly to bounding box coordinates and class probabilities (Source: 1506.02640v5.pdf - Chunk 1). Unlike traditional methods that use complex pipelines, YOLO uses a single convolutional network to simultaneously predict multiple bounding boxes and class probabilities (Source: 1506.02640v5.pdf - Chunk 3).
   [Retrieved 5 chunks | top sim: 7.6309]
Q: How d

In [30]:
# ── Step 4d: Test out-of-scope refusal ────────────────────────────────────

# ── TODO: Ask something that is clearly NOT in your document ─────────────
OUT_OF_SCOPE_Q = "What should I eat today"  # TODO: e.g. "What happened last year?" or a competitor topic

assert OUT_OF_SCOPE_Q.strip(), "Set OUT_OF_SCOPE_Q"

result = my_rag(OUT_OF_SCOPE_Q)
print(f"Q: {result['question']}")
print(f"A: {result['answer']}")
print()
print("✅ Did the model correctly refuse instead of hallucinating? (check above)")

Q: What should I eat today
A: I don't have enough information in the provided context to answer this question.

✅ Did the model correctly refuse instead of hallucinating? (check above)


---
## Step 5 — Structured Citation Output

**Your task:**
- Implement `my_cited_rag()` that returns a **JSON object** with `answer`, `citations`, and `has_sufficient_context` fields
- Run it on 2 of your 5 questions
- Parse and display the citations cleanly

**Expected JSON shape:**
```json
{
  "answer": "Revenue grew 23% ...",
  "citations": [
    {"claim": "Revenue grew 23%", "source": "my_doc.pdf", "section": "executive_summary"}
  ],
  "has_sufficient_context": true
}
```

In [31]:
# ── Step 5: Cited RAG with structured JSON output ─────────────────────────
import json

# ── TODO: Write a citation system prompt ─────────────────────────────────
# It must:
#   - Instruct the model to return ONLY a JSON object (no markdown, no prose)
#   - Define the exact JSON schema: {answer, citations: [{claim, source, section}], has_sufficient_context}
#   - Instruct the model to only include claims that are in the context
MY_CITATION_SYSTEM = """
Your task is to act as an expert research assistant.

You will answer questions based *only* on the provided [CONTEXT] and respond *only* with a JSON object. Do not include any prose or markdown outside the JSON.

The JSON object MUST conform to the following schema:
{
  "answer": "<Your concise answer based on context, if available>",
  "citations": [
    {
      "claim": "<A specific claim from your answer>",
      "source": "<source_name>",
      "section": "<section_tag>"
    }
  ], # Array of all claims made, each with its source and section.
  "has_sufficient_context": <true/false> # True if the context was sufficient to answer the question, False otherwise.
}

Rules:
1. If you cannot answer the question from the [CONTEXT], set `answer` to 'I don't have enough information in the provided context to answer this question.', set `citations` to an empty array `[]`, and `has_sufficient_context` to `false`.
2. For each claim in your `answer`, create a corresponding object in the `citations` array. The `claim` field should be a short, direct quote or paraphrase from your answer that is directly supported by the context.
3. The `source` and `section` for each citation MUST come directly from the metadata provided in the [CONTEXT].
4. Do NOT make up claims, sources, or sections.
5. Ensure the JSON is valid and complete, with no extra characters.
"""

def my_cited_rag(question: str, k: int = 5) -> dict:
    """
    RAG that returns a structured JSON response with citations.
    Falls back gracefully if JSON parsing fails.
    """
    # ── TODO: implement ───────────────────────────────────────────────────
    # Steps:
    #   1. hits    = my_hybrid_search(question, k=k)
    #   2. context = my_format_context(hits)
    #   3. messages = [{MY_CITATION_SYSTEM}, {user: context + question}]
    #   4. raw    = chat(messages)
    #   5. parse JSON from raw (strip ```json fences if present)
    #   6. return parsed dict (or {answer: raw, citations: [], parse_error: True} on failure)

    hits = my_hybrid_search(question, k=k)
    context = my_format_context(hits)

    messages = [
        {"role": "system", "content": MY_CITATION_SYSTEM},
        {"role": "user", "content": f"[CONTEXT]\n{context}\n\n[QUESTION]\n{question}"},
    ]

    raw_response = chat(messages)

    # Attempt to parse the JSON, handling potential markdown fences
    try:
        # Remove markdown code fences if present
        if raw_response.strip().startswith('```json') and raw_response.strip().endswith('```'):
            raw_response = raw_response.strip()[len('```json'):-len('```')].strip()
        parsed_response = json.loads(raw_response)
        if not isinstance(parsed_response, dict):
            raise ValueError("JSON response is not a dictionary")
        return parsed_response
    except (json.JSONDecodeError, ValueError) as e:
        return {
            "answer": raw_response,
            "citations": [],
            "has_sufficient_context": False,
            "parse_error": True,
            "error_message": str(e)
        }


# ── Run on your first 2 questions ────────────────────────────────────────
for q in MY_QUESTIONS[:2]:
    result = my_cited_rag(q)
    print(f"Q: {q}")
    print(f"A: {result.get('answer', 'N/A')}")
    print(f"Sufficient context: {result.get('has_sufficient_context')}")
    for c in result.get('citations', []):
        print(f"  • '{c.get('claim','')[:60]}' → {c.get('source','')} [{c.get('section','')}]")
    print()

Q: What is the reported processing speed of the YOLO model?
A: The YOLO model processes images at 155 frames per second.
Sufficient context: True
  • 'YOLO processes an astounding 155 frames per second' → 1506.02640v5.pdf [introduction]

Q: What is the main innovation introduced by the YOLO framework?
A: The main innovation of the YOLO framework is reframing object detection as a single regression problem, mapping image pixels directly to bounding box coordinates and class probabilities, rather than using complex, multi-component pipelines.
Sufficient context: True
  • 'YOLO reframes object detection as a single regression proble' → 1506.02640v5.pdf [general]
  • 'YOLO is a unified model that avoids complex pipelines by fra' → 1506.02640v5.pdf [methodology]



---
## Step 6 — Demonstrate 2 Failure Modes (and Their Fixes)

**Your task:** Choose any **2** of the 4 failure modes below. For each one:
1. Show the **broken** version (incorrect or missing output)
2. Show the **fixed** version (correct output)
3. Write a 1–2 sentence explanation of *why* the fix works

| Failure mode | How to trigger it |
|---|---|
| Hallucination | Use a weak system prompt + ask out-of-scope question |
| Context overflow | Set k to the total number of chunks |
| Format error | Ask for JSON without providing a schema |
| Retrieval failure | Ask with an acronym or synonym that BM25 cannot match |

In [32]:
# ── Step 6a: Failure Mode 1 ───────────────────────────────────────────────
# ── TODO: Choose one failure mode and demonstrate it ─────────────────────

FAILURE_1_NAME = "Hallucination"

print(f"FAILURE MODE 1: {FAILURE_1_NAME}")
print("="*60)
print()

# Broken: Demonstrate hallucination with a weak system prompt and an out-of-scope question.
print("❌ Broken (Weak System Prompt - Hallucination):")
WEAK_SYSTEM_PROMPT = "You are a helpful assistant. Answer the question." # No grounding/refusal rules
OUT_OF_SCOPE_Q = "What should I eat today?"

# Retrieve some (irrelevant) context for the out-of-scope question
hits_irrelevant = my_hybrid_search(OUT_OF_SCOPE_Q, k=1)
context_irrelevant = my_format_context(hits_irrelevant)

# Construct messages with a weak system prompt and the irrelevant context
messages_broken = [
    {"role": "system", "content": WEAK_SYSTEM_PROMPT},
    {"role": "user",   "content": f"[CONTEXT]\n{context_irrelevant}\n\n[QUESTION]\n{OUT_OF_SCOPE_Q}"},
]
hallucinated_answer = chat(messages_broken)
print(f"Q: {OUT_OF_SCOPE_Q}")
print(f"A: {hallucinated_answer}")
print()

# Fixed: Demonstrate refusal with the strong MY_SYSTEM_PROMPT (used by my_rag).
print("✅ Fixed (Strong Grounding System Prompt - Refusal):")
result_fixed = my_rag(OUT_OF_SCOPE_Q)
print(f"Q: {result_fixed['question']}")
print(f"A: {result_fixed['answer']}")

print()
# ── TODO: Explain why the fix works ──────────────────────────────────────
EXPLANATION_1 = "The weak system prompt allowed the model to invent an answer for an out-of-scope question, leading to hallucination. The strong system prompt, with explicit grounding and refusal rules ('Answer ONLY using the [CONTEXT] below'), correctly instructs the model to refuse to answer when the information is not present in the provided context." # Replaced placeholder
print(f"💡 Why the fix works: {EXPLANATION_1}")


FAILURE MODE 1: Hallucination

❌ Broken (Weak System Prompt - Hallucination):
Q: What should I eat today?
A: Based on the provided context, which discusses the methodology for object detection bounding boxes and confidence scores, there is no information regarding food or dietary recommendations. Therefore, I cannot provide a suggestion on what you should eat today.

✅ Fixed (Strong Grounding System Prompt - Refusal):
Q: What should I eat today?
A: I don't have enough information in the provided context to answer this question.

💡 Why the fix works: The weak system prompt allowed the model to invent an answer for an out-of-scope question, leading to hallucination. The strong system prompt, with explicit grounding and refusal rules ('Answer ONLY using the [CONTEXT] below'), correctly instructs the model to refuse to answer when the information is not present in the provided context.


In [33]:
# ── Step 6b: Failure Mode 2 ───────────────────────────────────────────────
# ── TODO: Choose a DIFFERENT failure mode ────────────────────────────────

FAILURE_2_NAME = "Retrieval Failure (Synonyms with BM25)"

assert FAILURE_2_NAME != FAILURE_1_NAME, "Choose a different failure mode for Step 6b"

print(f"FAILURE MODE 2: {FAILURE_2_NAME}")
print("="*60)
print()

# Broken: Query using a synonym/paraphrase that BM25 struggles with.
print("❌ Broken (BM25 with Synonym/Paraphrase):")
query_broken = "Explain intersection over union." # Document uses 'IOU' frequently
print(f"Query: '{query_broken}' (BM25 Search)")
broken_bm25_hits = my_bm25_search(query_broken, k=3)
for i, hit in enumerate(broken_bm25_hits):
    print(f"  Rank #{i+1} | score={hit['score']:.4f} | section={hit['metadata'].get('section')}")
    print(f"  {hit['text'][:140]}...")
print("\nAnswer using broken retrieval (via my_rag):")
result_broken_rag = my_rag(query_broken)
print(f"A: {result_broken_rag['answer']}")
print()

# Fixed: Rephrase the query with terms directly from the document or use hybrid search.
print("✅ Fixed (Rephrased Query with Hybrid Search):")
query_fixed = "What is IOU in object detection?" # Uses direct acronym 'IOU' and key concept 'object detection'
print(f"Query: '{query_fixed}' (Hybrid Search)")
fixed_hybrid_hits = my_hybrid_search(query_fixed, k=3)
for i, hit in enumerate(fixed_hybrid_hits):
    print(f"  Rank #{i+1} | rrf_score={hit['rrf_score']:.4f} | section={hit['metadata'].get('section')}")
    print(f"  {hit['text'][:140]}...")
print("\nAnswer using fixed retrieval (via my_rag):")
result_fixed_rag = my_rag(query_fixed)
print(f"A: {result_fixed_rag['answer']}")
print()

EXPLANATION_2 = "BM25 (keyword-based search) struggles when the query uses synonyms or different phrasing (e.g., 'intersection over union' vs. 'IOU'). By rephrasing the question with key terms from the document and leveraging hybrid search (which combines keyword and semantic understanding), retrieval accuracy significantly improves, leading to a more relevant answer." # Replaced placeholder
print(f"💡 Why the fix works: {EXPLANATION_2}")


FAILURE MODE 2: Retrieval Failure (Synonyms with BM25)

❌ Broken (BM25 with Synonym/Paraphrase):
Query: 'Explain intersection over union.' (BM25 Search)
  Rank #1 | score=10.7454 | section=methodology
  for those boxes. These conﬁdence scores reﬂect how conﬁdent the model is that the box contains an object and
also how accurate it thinks the...
  Rank #2 | score=2.5629 | section=general
  o perform localization and adapt that localizer to perform detection [32]. OverFeat efﬁciently performs slid-
ing window detection but it is...
  Rank #3 | score=2.5252 | section=general
  izations.
3. Comparison to Other Detection Systems Object detection is a core problem in computer vision.
Detection pipelines generally star...

Answer using broken retrieval (via my_rag):
A: Intersection over union (IOU) is used to define confidence scores, where the confidence is defined as $Pr(Object) * IOU_{truth}^{pred}$ (Source: 1506.02640v5.pdf - Chunk 1). If an object exists in a cell, the confidence score s

---
## Step 7 — Reflection

Answer the 3 short-answer questions below. Write directly in the markdown cell (double-click to edit).

### ✍️ Reflection Questions

**Q1: What chunk size did you choose and why was it appropriate for your document type?**

> *I choose this chunk size to mostly preserve context of the paper and of course for efficiency*

---

**Q2: Did hybrid search return noticeably different results than pure dense search on any of your queries? Describe one example.**

> *Both results output the same sentence*

---

**Q3: If you were deploying this RAG system to real users, what one improvement would you prioritise first and why?**

> *The top priority would be the context document must be formatted correctly or it would poorly search the context*

In [34]:
# ── Final submission check ────────────────────────────────────────────────
# Run this cell last. All assertions must pass before submitting.

checks = []

# Step 0
checks.append(("Document loaded",          len(MY_DOCUMENT.strip()) >= 500))
checks.append(("DOC_NAME set",             bool(DOC_NAME.strip())))

# Step 1
checks.append(("Chunks created",           my_chunks is not None and len(my_chunks) >= 3))
checks.append(("Chunk size set",           CHUNK_SIZE > 0 and CHUNK_OVERLAP > 0))

# Step 2
checks.append(("Vectors created",          my_vectors is not None and len(my_vectors) == len(my_chunks)))
checks.append(("ChromaDB indexed",         my_collection.count() == len(my_chunks)))
checks.append(("dense_search working",     bool(my_dense_search(MY_QUESTIONS[0], k=1))))

# Step 3
checks.append(("BM25 index built",         my_bm25_index is not None))
checks.append(("bm25_search working",      bool(my_bm25_search(MY_QUESTIONS[0], k=1))))
checks.append(("hybrid_search working",    bool(my_hybrid_search(MY_QUESTIONS[0], k=1))))

# Step 4
checks.append(("System prompt written",    len(MY_SYSTEM_PROMPT.strip()) > 50))
checks.append(("5 questions filled",       all(q.strip() for q in MY_QUESTIONS)))
checks.append(("my_rag() works",           bool(my_rag(MY_QUESTIONS[0]).get('answer'))))

# Step 5
checks.append(("Citation system prompt",   len(MY_CITATION_SYSTEM.strip()) > 50))

# Step 6
checks.append(("Failure mode 1 named",     bool(FAILURE_1_NAME.strip())))
checks.append(("Failure mode 2 named",     bool(FAILURE_2_NAME.strip())))
checks.append(("Different failure modes",  FAILURE_1_NAME != FAILURE_2_NAME))

# Results
print("\n📋 Submission Checklist")
print("=" * 50)
all_pass = True
for name, result in checks:
    icon = "✅" if result else "❌"
    print(f"  {icon}  {name}")
    if not result:
        all_pass = False

print()
if all_pass:
    print("🎉 All checks passed! Your notebook is ready to submit.")
    print(f"   Save as: RAG_Assignment_{DOC_TOPIC.replace(' ', '_')[:30]}.ipynb")
else:
    print("⚠️  Some checks failed. Fix the ❌ items above before submitting.")


📋 Submission Checklist
  ✅  Document loaded
  ✅  DOC_NAME set
  ✅  Chunks created
  ✅  Chunk size set
  ✅  Vectors created
  ✅  ChromaDB indexed
  ✅  dense_search working
  ✅  BM25 index built
  ✅  bm25_search working
  ✅  hybrid_search working
  ✅  System prompt written
  ✅  5 questions filled
  ✅  my_rag() works
  ✅  Citation system prompt
  ✅  Failure mode 1 named
  ✅  Failure mode 2 named
  ✅  Different failure modes

🎉 All checks passed! Your notebook is ready to submit.
   Save as: RAG_Assignment_You_Only_Look_Once.ipynb
